<h6><center>Big Data : Enjeux, stockage et extraction</center></h6>

<h1>
<hr style=" border:none; height:3px;">
<center>Examen Big Data : Spark</center>
<hr style=" border:none; height:3px;">
</h1>
Février 2026

In [1]:
from pyspark import SparkContext, SparkConf
import os

seed_value = 0
os.environ['PYTHONHASHSEED'] = str(seed_value)

In [2]:
## Configuration

monAppli = "ExamenSpark"

monCluster = "local[*]"

conf = SparkConf().setAppName(monAppli).setMaster(monCluster)

## Instanciation

sc = SparkContext.getOrCreate(conf = conf)

print("Initialisation réussie !")

Initialisation réussie !


## Exercice n°1 : Digrammes

On souhaite calculer le nombre d'occurrences par couple de caractères qui se suivent dans un mot. Par exemple, pour le mot "hello", la sortie attendue est :   
`he -> 1`   
`el -> 1`   
`ll -> 1`    
`lo -> 1`    
Pour cela, on va procéder par étape. Ci-dessous, un exemple qui vous permettra de tester votre code :

In [3]:
string = ["Hey Leo"]

1. Produire le RDD `rdd1` dont les éléments ont la forme suivante : `[(mot, position), caractère]`. Par exemple, pour le texte "Hey Leo", le RDD attendu est le suivant :  
```
[
[(hey, 1),h],
[(hey, 2),e],
[(hey, 3),y],
[(leo, 1),l],
[(leo, 2),e],
[(leo, 3),o]
]
```

**Remarque :**
- La fonction python `enumerate` appliquée à une liste (ou une chaîne de cractères) permet d'obtenir, pour chaque élément de la liste (ou de la chaîne de caractères), son indice et sa valeur. Par exemple :

In [4]:
maListe = ['Fraise','Pomme','Banane']
for indice, valeur in enumerate(maListe) :
  print(indice, valeur)

0 Fraise
1 Pomme
2 Banane


- La méthode `.lower()` appliquée à une chaîne de caractères permet le passage en minuscule.
- La méthode `.split()` appliquée à une chaîne de caractères permet de découper la chaîne de caractères en fonction des espaces. Par exemple :

In [5]:
print("La vie est belle".split())

['La', 'vie', 'est', 'belle']


In [6]:
# Version `lambda` fonction

rdd1 = sc.parallelize(string)\
        .flatMap(lambda s: s.lower().split())\
        .flatMap(lambda w: [[(w,i + 1),l] for i,l in enumerate(w)])
rdd1.collect()

[[('hey', 1), 'h'],
 [('hey', 2), 'e'],
 [('hey', 3), 'y'],
 [('leo', 1), 'l'],
 [('leo', 2), 'e'],
 [('leo', 3), 'o']]

In [7]:
# Version `def` fonction

def mapper(word):
  paires = []
  for indice, lettre in enumerate(word):
    paire = [(word,indice + 1),lettre]
    paires.append(paire)
  return paires

rdd1 = sc.parallelize(string)\
        .flatMap(lambda s: s.lower().split())\
        .flatMap(mapper)
rdd1.collect()

[[('hey', 1), 'h'],
 [('hey', 2), 'e'],
 [('hey', 3), 'y'],
 [('leo', 1), 'l'],
 [('leo', 2), 'e'],
 [('leo', 3), 'o']]

2. À partir du RDD précédent, construire le RDD `rdd2` suivant, dont les indices ont été décalés d'une position :    
```
[
[(hey, 0),h],
[(hey, 1),e],
[(hey, 2),y],
[(leo, 0),l],
[(leo, 1),e],
[(leo, 2),o]
]
```
Si vous n'avez pas réussi la question précédente, décommentez et utilisez la liste ci-dessous.

In [ ]:
#rdd1 = [[('hey', 1), 'h'],
#        [('hey', 2), 'e'],
#        [('hey', 3), 'y'],
#        [('leo', 1), 'l'],
#        [('leo', 2), 'e'],
#        [('leo', 3), 'o']]

In [ ]:
#rdd1 = sc.parallelize(rdd1)

In [8]:
rdd2 = rdd1.map(lambda paire: [(paire[0][0],paire[0][1] - 1),paire[1]])
rdd2.collect()

[[('hey', 0), 'h'],
 [('hey', 1), 'e'],
 [('hey', 2), 'y'],
 [('leo', 0), 'l'],
 [('leo', 1), 'e'],
 [('leo', 2), 'o']]

3. Joindre les deux RDD `rdd1` et `rdd2` pour obtenir le RDD `rdd3` suivant :
```
[
[(hey, 1),(h, e)],
[(hey, 2),(e, y)],
[(leo, 1),(l, e)],
[(leo, 2),(e, o)]
]
```
On joint donc deux lettres successives puisqu'un des indices a été décalé.

Si vous n'avez pas réussi la question précédente, décommentez et utilisez la liste ci-dessous.

In [ ]:
#rdd2 = [[('hey', 0), 'h'],
#        [('hey', 1), 'e'],
#        [('hey', 2), 'y'],
#        [('leo', 0), 'l'],
#        [('leo', 1), 'e'],
#        [('leo', 2), 'o']]

In [ ]:
#rdd2 = sc.parallelize(rdd2)

In [9]:
rdd3 = rdd1.join(rdd2)
rdd3.collect()

[(('leo', 2), ('e', 'o')),
 (('leo', 1), ('l', 'e')),
 (('hey', 2), ('e', 'y')),
 (('hey', 1), ('h', 'e'))]

4. À partir de `rdd3`, produire le RDD `rdd4` suivant, qui associe à chaque couple de lettres successives la valeur 1 :
```
[
[(h, e),1],
[(e, y),1],
[(l, e),1],
[(e, o),1]
]
```

Si vous n'avez pas réussi la question précédente, décommentez et utilisez la liste ci-dessous.

In [ ]:
#rdd3 = [(('leo', 2), ('e', 'o')),
#        (('leo', 1), ('l', 'e')),
#        (('hey', 2), ('e', 'y')),
#        (('hey', 1), ('h', 'e'))]

In [ ]:
#rdd3 = sc.parallelize(rdd3)

In [10]:
rdd4 = rdd3.map(lambda paire: (paire[1],1))
rdd4.collect()

[(('e', 'o'), 1), (('l', 'e'), 1), (('e', 'y'), 1), (('h', 'e'), 1)]

5. À partir de `rdd4`, calculer le nombre d'occurrences de chaque couple de caractères. Pour notre exemple, le résultat sera identique au résultat de la question précédente :
```
[
[(h, e),1],
[(e, y),1],
[(l, e),1],
[(e, o),1]
]
```
Si vous n'avez pas réussi la question précédente, décommentez et utilisez la liste ci-dessous.

In [ ]:
#rdd4 = [(('e', 'o'), 1),
#        (('l', 'e'), 1),
#        (('e', 'y'), 1),
#        (('h', 'e'), 1)]

In [ ]:
#rdd4 = sc.parallelize(rdd4)

In [11]:
rdd4.reduceByKey(lambda x, y: x + y).collect()

[(('e', 'y'), 1), (('h', 'e'), 1), (('e', 'o'), 1), (('l', 'e'), 1)]

6. Calculez le nombre d'occurrences de chaque couple de caractères dans le texte `Souvenirs_de_Sherlock_Holmes.txt` et retournez les 10 couples les plus fréquents.

**Remarque :** Attention aux chaînes de caractères vides après extraction des mots constituant les phrases.




In [ ]:
# Ce qui change
sherlock = sc.textFile("Souvenirs_de_Sherlock_Holmes.txt")

In [ ]:
# Question 1
# Ce qui change : ajout de `filter()`
rdd1 = sherlock.flatMap(lambda s: s.lower().split())\
               .filter(lambda w: len(w) > 0)\
               .flatMap(lambda w: [[(w,i + 1),l] for i,l in enumerate(w)])

# Question 2
rdd2 = rdd1.map(lambda paire: [(paire[0][0],paire[0][1] - 1), paire[1]])

# Question 3
rdd3 = rdd1.join(rdd2)

# Question 4
rdd4 = rdd3.map(lambda paire: (paire[1],1))

# Question 5
# Ce qui change : ajout de `sortBy()` et `take()`
rdd4.reduceByKey(lambda x, y: x + y)\
    .sortBy(lambda x: x[1], ascending=False)\
    .take(10)


## Exercice n°2 : k-mer

**Correction :** Cet exercice était plus difficile, car moins guidé, que l'exercice précédent. Personne n'y a répondu, la notation ne portera donc que sur l'exercice précédent.

Le terme **k-mer** désigne toutes les sous-chaînes possibles de longueur k contenues dans une chaîne. En génomique informatique, les k-mers désignent toutes les sous-séquences possibles (de longueur k) à partir d'une lecture obtenues par séquençage d'ADN.

Soient plusieurs fichiers contenant chacun un séquençage d'ADN, caractérisé par une suite de lettres (sans espace). Écrire le code Spark permettant de calculer le nombre de fois qu'apparaît une suite de 3 lettres **dans chacun de ces fichiers**. Ne pas utiliser de boucle `for` pour traiter chacun des fichiers.

**Exemple :** Supposons le fichier d'entrée contenant : `ACACACAGT`. Le résultat retourné doit être : `(ACA,3)`, `(CAC,2)`, `(CAG,1)`, `(AGT,1)`.

In [ ]:
rdd = sc.wholeTextFiles("./adn")
rdd.take(1)
#rdd = sc.parallelize(["ACACACAGT"])

In [ ]:
# Version `lambda` fonction

# flatMap :
#    clé => nom du fichier
#    valeur => 3-mer (sous-chaîne de longueur 3)
# map :
#    clé => paire (nom du fichier, 3-mer)
#    valeur => 1
# reduceByKey : somme des 1 par clé (nom du fichier, 3-mer)

rdd.flatMap(lambda x: [(x[0], x[1][i:i+3]) for i in range(len(x[1])-2)])\
   .map(lambda x: (x,1))\
   .reduceByKey(lambda x,y : x + y)\
   .collect()

In [ ]:
# Version `def` fonction

def mapper(adn):
  kmers = []
  for i in range(len(adn[1]) - 2):
    # Parcourir toutes les combinaisons de 3 lettres successives
    # Clé : nom du fichier
    # Valeur : sous-chaîne de longueur 3
    paire = (adn[0], adn[1][i:i+3])
    kmers.append(paire)
  return kmers

rdd.flatMap(mapper)\
   .map(lambda x: (x,1))\
   .reduceByKey(lambda x,y : x + y)\
   .collect()